In [1]:
# ============================================
# 1. IMPORTS
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

In [2]:
# ============================================
# 2. CARGAR CSV MAESTROS
# ============================================

tracks = pd.read_csv("CSV_MASTER/all_tracks.csv")

# Partículas reconstruidas
muons = pd.read_csv("CSV_MASTER/all_muons.csv")
electrons = pd.read_csv("CSV_MASTER/all_electrons.csv")
photons = pd.read_csv("CSV_MASTER/all_photons.csv")
taus = pd.read_csv("CSV_MASTER/all_taus.csv")

# Hadrons si existe
try:
    hadrons = pd.read_csv("CSV_MASTER/all_hadrons.csv")
except:
    hadrons = pd.DataFrame(columns=["track_id"])  # crea vacío si no existe



In [5]:
# ============================================
# 3. ETIQUETAR TRACKS POR PARTÍCULA VERDADERA
# ============================================

# Buscamos una columna de track_id (ajusta si tu CSV usa otro nombre)
possible_track_cols = ["event_id", "TrackID", "trk_id"]

def detect_track_column(df):
    for col in possible_track_cols:
        if col in df.columns:
            return col
    return None

track_id_col = detect_track_column(tracks)

if track_id_col is None:
    raise ValueError("No se encontró columna track_id en all_tracks.csv")

print("Columna de track encontrada:", track_id_col)

# --- Función para etiquetar ---
def label_tracks(tracks_df, particle_df, label):
    col = detect_track_column(particle_df)
    if col is None:
        print(f"[WARN] {label} no tiene columna track_id, ignorado")
        return

    ids = set(particle_df[col].unique())
    tracks_df.loc[tracks_df[track_id_col].isin(ids), "label"] = label


# Crear columna label única
tracks["label"] = "unknown"

# Etiquetar
label_tracks(tracks, muons, "muon")
label_tracks(tracks, electrons, "electron")
label_tracks(tracks, photons, "photon")
label_tracks(tracks, taus, "tau")
label_tracks(tracks, hadrons, "hadron")

# Filtrar solo tracks con etiqueta verdadera
tracks = tracks[tracks["label"] != "unknown"]

print(f"Total de tracks etiquetados: {len(tracks)}")

Columna de track encontrada: event_id
[WARN] hadron no tiene columna track_id, ignorado
Total de tracks etiquetados: 2993


In [6]:
# ============================================
# 4. PREPARAR X,y PARA ML
# ============================================

# Quitar columnas no numéricas que no sirven como features
non_features = ["label", "event_id"]

for col in tracks.columns:
    if tracks[col].dtype == "object" and col not in non_features:
        non_features.append(col)

y = tracks["label"]
X = tracks.drop(columns=non_features)

# Codificar clases
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Normalizar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [7]:
# ============================================
# 5. TRAIN/TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.25,
    random_state=42,
    stratify=y_encoded
)


# ============================================
# 6. ENTRENAR MODELO (UNA VEZ)
# ============================================

model = LogisticRegression(
    multi_class="multinomial",
    max_iter=1000,
    solver="lbfgs"
)

model.fit(X_train, y_train)


C:\Users\Usuario\AppData\Roaming\Python\Python312\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)

In [ ]:
# ============================================
# 7. REPORTES
# ============================================

y_pred = model.predict(X_test)

print("\n===== CLASSIFICATION REPORT =====\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7,5))
plt.imshow(cm, cmap="Blues")
plt.title("Matriz de Confusión")
plt.xlabel("Predicción")
plt.ylabel("Verdadero")

plt.xticks(np.arange(len(le.classes_)), le.classes_, rotation=45)
plt.yticks(np.arange(len(le.classes_)), le.classes_)

# Añadir números a la matriz
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# 8. CURVAS ROC MULTICLASE
# ============================================

y_proba = model.predict_proba(X_test)
n_classes = len(le.classes_)

plt.figure(figsize=(8,6))

for i in range(n_classes):
    fpr, tpr, _ = roc_curve((y_test == i).astype(int), y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{le.classes_[i]} (AUC={roc_auc:.3f})")

plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Multiclase")
plt.legend()
plt.show()


In [ ]:

# ============================================
# 9. CURVAS DE APRENDIZAJE
# ============================================

train_sizes, train_scores, test_scores = learning_curve(
    model, X_scaled, y_encoded, cv=5
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(8,5))
plt.plot(train_sizes, train_mean, marker="o", label="Train Score")
plt.plot(train_sizes, test_mean, marker="o", label="Test Score")

plt.xlabel("Tamaño del entrenamiento")
plt.ylabel("Accuracy")
plt.title("Curva de aprendizaje — Logistic Regression")
plt.legend()
plt.grid()
plt.show()